# Complete PRE/POST qEEG response-association grid

## Objective

The published response analysis used four selected condition/state/feature cells. This notebook fills the complete grid: **PRE and POST × awake and sleep × DFA, entropy, and PLI**. For every cell, it tests both one clip at a time and the mean of the two matching clips from each patient.

The goal is to show every measured association, including negative results, rather than report only the combinations selected in the source paper.

In [1]:
from __future__ import annotations

import importlib
import os
import sys
from pathlib import Path

from IPython.display import display

SEED = 20260912
PERMUTATIONS = 1_000_000
REPO = Path(os.environ.get("IESSEEG_BASELINES_REPO", Path.cwd())).resolve()
METADATA_CSV = Path(os.environ["IESSEEG_METADATA_CSV"])
RAW_FEATURES_DIR = Path(os.environ["IESSEEG_RAJARAMAN_RAW_FEATURES_DIR"])

if not METADATA_CSV.is_file():
    raise FileNotFoundError("IESSEEG_METADATA_CSV does not point to a file")
if not RAW_FEATURES_DIR.is_dir():
    raise NotADirectoryError("IESSEEG_RAJARAMAN_RAW_FEATURES_DIR does not point to a directory")

analysis_dir = REPO / "analysis" / "response_features"
sys.path.insert(0, str(analysis_dir))
run_analysis = importlib.import_module("analyze_full_qeeg_response_grid").run_analysis

print(f"Configuration loaded: {PERMUTATIONS:,} patient-level permutations; seed {SEED}.")

Configuration loaded: 1,000,000 patient-level permutations; seed 20260912.


## Experimental design

Each condition/state cell contains two clips from each of 50 patients. Immediate response contains 32 responders and 18 non-responders; sustained response contains 28 responders and 22 non-responders.

For a single-clip test, both clips contribute values, but the significance test keeps the two values from one patient together. For a patient-average test, the two matching clips are averaged first. The raw P-value tests whether response labels and EEG values are unrelated across patients. Holm correction covers all 24 grid tests within each endpoint.

In [2]:
grid = run_analysis(
    METADATA_CSV,
    RAW_FEATURES_DIR,
    n_permutations=PERMUTATIONS,
    seed=SEED,
)
assert len(grid) == 48
assert set(grid.groupby("endpoint").size()) == {24}
print("Complete grid verified: 24 tests for each response endpoint.")

Complete grid verified: 24 tests for each response endpoint.


## Results

`Separation AUROC` is always at least 0.5 and shows the strength of separation in the better direction. `Direction` states whether responders have higher or lower values. The final column is the decision after correcting all 24 comparisons for that endpoint.

In [3]:
columns = {
    "input": "EEG",
    "quantity": "Quantity",
    "aggregation": "Calculation unit",
    "source_selected_cell": "Used by source paper",
    "separation_auc": "Separation AUROC",
    "responder_direction": "Direction",
    "patient_clustered_permutation_p": "Raw P",
    "holm_p_across_24_grid_tests": "Holm P",
    "significant_after_holm_0_05": "Significant",
}
for endpoint in ("immediate", "sustained"):
    print(f"{endpoint.title()} response")
    view = grid.loc[grid.endpoint.eq(endpoint), list(columns)].rename(columns=columns)
    display(
        view.style.format(
            {
                "Separation AUROC": "{:.3f}",
                "Raw P": "{:.6g}",
                "Holm P": "{:.6g}",
            }
        )
    )

Immediate response


,EEG,Quantity,Calculation unit,Used by source paper,Separation AUROC,Direction,Raw P,Holm P,Significant
0,PRE awake,beta DFA intercept,one clip,True,0.566,lower,0.411669,1,False
1,PRE awake,beta DFA intercept,mean of two awake clips,True,0.587,lower,0.319591,1,False
2,PRE awake,beta Shannon entropy,one clip,False,0.557,higher,0.463058,1,False
3,PRE awake,beta Shannon entropy,mean of two awake clips,False,0.562,higher,0.476341,1,False
4,PRE awake,delta PLI connectivity,one clip,True,0.614,lower,0.153661,1,False
5,PRE awake,delta PLI connectivity,mean of two awake clips,True,0.607,lower,0.218102,1,False
6,PRE sleep,beta DFA intercept,one clip,False,0.505,lower,0.95624,1,False
7,PRE sleep,beta DFA intercept,mean of two sleep clips,False,0.500,higher,1,1,False
8,PRE sleep,beta Shannon entropy,one clip,False,0.653,higher,0.0506599,1,False
9,PRE sleep,beta Shannon entropy,mean of two sleep clips,False,0.661,higher,0.0596659,1,False


Sustained response


,EEG,Quantity,Calculation unit,Used by source paper,Separation AUROC,Direction,Raw P,Holm P,Significant
24,PRE awake,beta DFA intercept,one clip,True,0.655,lower,0.042051,0.756917,False
25,PRE awake,beta DFA intercept,mean of two awake clips,True,0.683,lower,0.026276,0.525519,False
26,PRE awake,beta Shannon entropy,one clip,False,0.548,higher,0.522764,1,False
27,PRE awake,beta Shannon entropy,mean of two awake clips,False,0.544,higher,0.607329,1,False
28,PRE awake,delta PLI connectivity,one clip,True,0.666,lower,0.029587,0.562152,False
29,PRE awake,delta PLI connectivity,mean of two awake clips,True,0.657,lower,0.0567649,0.965004,False
30,PRE sleep,beta DFA intercept,one clip,False,0.615,lower,0.149721,1,False
31,PRE sleep,beta DFA intercept,mean of two sleep clips,False,0.615,lower,0.169835,1,False
32,PRE sleep,beta Shannon entropy,one clip,False,0.638,higher,0.0675809,1,False
33,PRE sleep,beta Shannon entropy,mean of two sleep clips,False,0.636,higher,0.102452,1,False


## Which results survive correction?

The table below contains only associations that remain below 0.05 after accounting for all 24 displayed grid tests. This filter is for reading convenience; the complete tables above remain the primary result.

In [4]:
significant = grid.loc[
    grid.significant_after_holm_0_05,
    [
        "endpoint",
        "input",
        "quantity",
        "aggregation",
        "source_selected_cell",
        "separation_auc",
        "responder_direction",
        "holm_p_across_24_grid_tests",
    ],
]
display(significant.reset_index(drop=True))

,endpoint,input,quantity,aggregation,source_selected_cell,separation_auc,responder_direction,holm_p_across_24_grid_tests
0,immediate,POST sleep,beta Shannon entropy,one clip,True,0.781250,higher,0.001920
1,immediate,POST sleep,beta Shannon entropy,mean of two sleep clips,True,0.822917,higher,0.002438
2,sustained,POST awake,beta DFA intercept,one clip,True,0.768669,lower,0.003024
3,sustained,POST awake,beta DFA intercept,mean of two awake clips,True,0.816558,lower,0.001650
4,sustained,POST sleep,beta Shannon entropy,one clip,True,0.802760,higher,0.000168
5,sustained,POST sleep,beta Shannon entropy,mean of two sleep clips,True,0.852273,higher,0.000230


## Paper-defined combined and change metrics

The complete catalog also includes every derived quantity from the paper that can be calculated from the reproduced features: R0, duration-adjusted P0, R1, duration-adjusted P1, PRE-to-POST changes in awake DFA, awake PLI, and sleep entropy, and the relapse metric rho.

R0 and R1 contain EEG only. P0 and P1 add the clinical duration category. Rho was defined for time to relapse, not binary response; its response association is displayed only to make the catalog complete and is labeled with its original target.

In [5]:
derived_module = importlib.import_module("analyze_paper_derived_response_metrics")
derived = derived_module.run_analysis(
    METADATA_CSV,
    RAW_FEATURES_DIR,
    n_permutations=PERMUTATIONS,
    seed=20260914,
)
catalog = derived_module.build_complete_catalog(grid, derived)
assert len(derived) == 32
assert len(catalog) == 80
print("Complete catalog verified: 48 individual-feature rows plus 32 paper-derived rows.")

Complete catalog verified: 48 individual-feature rows plus 32 paper-derived rows.


In [6]:
derived_columns = [
    "analysis_family",
    "source_target",
    "input",
    "quantity",
    "aggregation",
    "matches_source_patient_construction",
    "separation_auc",
    "responder_direction",
    "patient_clustered_permutation_p",
    "holm_p_across_16_derived_tests",
    "significant_after_holm_0_05",
]
for endpoint in ("immediate", "sustained"):
    print(f"{endpoint.title()} response: paper-derived quantities")
    view = derived.loc[derived.endpoint_tested_here.eq(endpoint), derived_columns]
    display(
        view.style.format(
            {
                "separation_auc": "{:.3f}",
                "patient_clustered_permutation_p": "{:.6g}",
                "holm_p_across_16_derived_tests": "{:.6g}",
            }
        )
    )

Immediate response: paper-derived quantities


,analysis_family,source_target,input,quantity,aggregation,matches_source_patient_construction,separation_auc,responder_direction,patient_clustered_permutation_p,holm_p_across_16_derived_tests,significant_after_holm_0_05
0,published EEG response score,sustained response,PRE awake,R0,one PRE awake clip,False,0.642,higher,0.0720769,0.360385,False
1,published duration-adjusted response probability,sustained response,PRE awake,P0,one PRE awake clip,False,0.711,higher,0.00953299,0.0953299,False
2,published EEG response score,sustained response,PRE awake,R0,mean of two PRE awake clips,True,0.661,higher,0.0597929,0.358758,False
3,published duration-adjusted response probability,sustained response,PRE awake,P0,mean of two PRE awake clips,True,0.715,higher,0.011552,0.0953299,False
4,published EEG response score,sustained response,POST awake + POST sleep,R1,all four within-patient POST awake/sleep clip pairs,False,0.798,higher,3.1e-05,0.000403,True
5,published duration-adjusted response probability,sustained response,POST awake + POST sleep,P1,all four within-patient POST awake/sleep clip pairs,False,0.817,higher,9.99999e-06,0.00016,True
6,published relapse metric,relapse time among immediate responders,POST awake + POST sleep,rho,all four within-patient POST awake/sleep clip pairs,False,0.779,lower,0.000122,0.001344,True
7,published EEG response score,sustained response,POST awake + POST sleep,R1,mean of two POST awake and two POST sleep clips,True,0.847,higher,1.8e-05,0.000252,True
8,published duration-adjusted response probability,sustained response,POST awake + POST sleep,P1,mean of two POST awake and two POST sleep clips,True,0.856,higher,1.5e-05,0.000225,True
9,published relapse metric,relapse time among immediate responders,POST awake + POST sleep,rho,mean of two POST awake and two POST sleep clips,True,0.819,lower,0.000112,0.001344,True


Sustained response: paper-derived quantities


,analysis_family,source_target,input,quantity,aggregation,matches_source_patient_construction,separation_auc,responder_direction,patient_clustered_permutation_p,holm_p_across_16_derived_tests,significant_after_holm_0_05
16,published EEG response score,sustained response,PRE awake,R0,one PRE awake clip,False,0.725,higher,0.002632,0.021016,True
17,published duration-adjusted response probability,sustained response,PRE awake,P0,one PRE awake clip,False,0.783,higher,0.000229,0.00229,True
18,published EEG response score,sustained response,PRE awake,R0,mean of two PRE awake clips,True,0.747,higher,0.002627,0.021016,True
19,published duration-adjusted response probability,sustained response,PRE awake,P0,mean of two PRE awake clips,True,0.784,higher,0.00046,0.00414,True
20,published EEG response score,sustained response,POST awake + POST sleep,R1,all four within-patient POST awake/sleep clip pairs,False,0.873,higher,9.99999e-07,1.6e-05,True
21,published duration-adjusted response probability,sustained response,POST awake + POST sleep,P1,all four within-patient POST awake/sleep clip pairs,False,0.893,higher,9.99999e-07,1.6e-05,True
22,published relapse metric,relapse time among immediate responders,POST awake + POST sleep,rho,all four within-patient POST awake/sleep clip pairs,False,0.860,lower,9.99999e-07,1.6e-05,True
23,published EEG response score,sustained response,POST awake + POST sleep,R1,mean of two POST awake and two POST sleep clips,True,0.924,higher,9.99999e-07,1.6e-05,True
24,published duration-adjusted response probability,sustained response,POST awake + POST sleep,P1,mean of two POST awake and two POST sleep clips,True,0.940,higher,9.99999e-07,1.6e-05,True
25,published relapse metric,relapse time among immediate responders,POST awake + POST sleep,rho,mean of two POST awake and two POST sleep clips,True,0.909,lower,2e-06,2.2e-05,True


## Interpretation boundary

The four cells marked as source-selected reproduce quantities chosen in the published sustained-response analysis. The other eight feature cells are newly examined here. The 24 individual-feature tests and 16 paper-derived tests are corrected as separate, explicitly named families. Rho remains a relapse-time metric even when its numerical association with a response label is displayed. Because the locally reproduced entropy has a constant absolute offset from the paper, P1 is useful here for ranking and association but its numerical probability is not calibrated. All results use the source cohort and do not estimate performance on a new clinical cohort.